# Benchmark RAG: ejecución de principio a fin

Reproduce todas las cifras del proyecto. **Requiere GPU libre (turno exclusivo)**: en total ~45-60 min.
Cada celda llama a los mismos scripts que usó el grupo; los resultados quedan en `sendoa/resultados_oficiales/` (Fase 2) y `nahia/resultados/` (Fase 3).

Orden: 0 comprobaciones → 1 Fase 2 (embeddings) → 2 Tabla A y ganadores → 3 Fase 3 (LLMs) → 4 puntuación a mano → 5 Tablas B/C y decisión.

In [9]:
import subprocess, sys, json, pathlib, shutil
RAIZ = pathlib.Path.cwd()
if not (RAIZ/'comun').exists(): RAIZ = RAIZ.parent
PY = str(RAIZ/'.venv'/'bin'/'python')
def sh(cmd, cwd=RAIZ):
    print('$', cmd); r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True); print(r.stdout[-3000:], r.stderr[-1500:] if r.returncode else ''); r.check_returncode()

## 0. Comprobaciones
Datos congelados (hashes que se guardan en cada `_meta.json`), GPU libre y umbrales.

In [10]:
sh('nvidia-smi | grep MiB | head -1')   # debe mostrar ~100 MiB en uso
sh(f'sha256sum comun/chunks_boe.json comun/chunks_debian.json comun/golden_set_boe.json comun/golden_set_debian.json comun/prompt.py')
print(open(RAIZ/'comun'/'umbrales.md').read())

$ nvidia-smi | grep MiB | head -1
|  0    1   0   0  |             102MiB / 20096MiB    | 42      0 |  3   0    2    0    0 |
 
$ sha256sum comun/chunks_boe.json comun/chunks_debian.json comun/golden_set_boe.json comun/golden_set_debian.json comun/prompt.py
97b4ae1fb58218ddbfc812119f7b417b1f951a478b1d966eb216149faf0c2faf  comun/chunks_boe.json
b30e886ccf694f8126ae4aa971347587298a9fd3f8be84350f2b5b0b1824a5e3  comun/chunks_debian.json
b48526d873b95572ecfd7d586e5dff2c97fe786d321182fcf9213120d2d39a53  comun/golden_set_boe.json
193a4d62d02ddb46c713574a9ab900e561bbb6415cb65a93951ca8ff9a82d40b  comun/golden_set_debian.json
bb2797270821d6b6017f99496de5658d9ce059cf04ebfa7a852ba2a35108f4ff  comun/prompt.py
 
# Umbrales de decisión (acordados por el grupo el 2026-11-22)

Se fijan ANTES de puntuar a mano y de rellenar las Tablas B y C. Se aplican por escenario; lo que no cumple se descarta aunque sea rápido o barato.

| Métrica | Umbral | Tabla |
|---|---|---|
| Recall@5 del embedding | ≥ 0,80 | A

## 1. Fase 2: embeddings (BOE y Debian)
Prefijos y longitud en `sendoa/config_*.json`. EmbeddingGemma se carga de la copia local `sendoa/modelos/embeddinggemma-300m`.

In [11]:
EMB = [('BAAI/bge-m3','sendoa/config_bge_m3.json'),('Qwen/Qwen3-Embedding-0.6B','sendoa/config_qwen3.json'),('sendoa/modelos/embeddinggemma-300m','sendoa/config_gemma.json')]
for modelo, cfg in EMB:
    for esc in ('boe','debian'):
        sh(f'{PY} sendoa/eval_retrieval.py --modelo {modelo} --escenario {esc} --chunks comun/chunks_{esc}.json --golden comun/golden_set_{esc}.json --config {cfg} --salida sendoa/resultados_oficiales --turno-exclusivo')

$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python sendoa/eval_retrieval.py --modelo BAAI/bge-m3 --escenario boe --chunks comun/chunks_boe.json --golden comun/golden_set_boe.json --config sendoa/config_bge_m3.json --salida sendoa/resultados_oficiales --turno-exclusivo
Modelo: BAAI/bge-m3 | escenario: boe
Recall@1=0.417  Recall@3=0.833  Recall@5=0.833  MRR=0.813  (n=12)
Indexado: 8.07 s | indice: 1.05 MB | chunks truncados: 0
Resultados: sendoa/resultados_oficiales/BAAI_bge-m3__boe__20261122-212534.json
 
$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python sendoa/eval_retrieval.py --modelo BAAI/bge-m3 --escenario debian --chunks comun/chunks_debian.json --golden comun/golden_set_debian.json --config sendoa/config_bge_m3.json --salida sendoa/resultados_oficiales --turno-exclusivo
Modelo: BAAI/bge-m3 | escenario: debian
Recall@1=0.167  Recall@3=0.500  Recall@5=0.583  MRR=0.508  (n=12)
Indexado: 41.02 s | indice: 4.51 MB | chunks truncados: 0
Resultados: sendoa/resultados_oficiale

## 2. Tabla A y ganadores
Criterio: mejor Recall@5; desempate por MRR. Los ganadores están en `nahia/ganadores_embedding.json` (BOE: Qwen3-Embedding; Debian: EmbeddingGemma, desempate por MRR).

In [12]:
sh(f'{PY} sendoa/tabla_a.py --resultados sendoa/resultados_oficiales --salida sendoa/tablas --solo-oficiales')
for e in ('boe','debian'): print(open(RAIZ/'sendoa'/'tablas'/f'tabla_a_{e}.md').read())
print(open(RAIZ/'nahia'/'ganadores_embedding.json').read())

$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python sendoa/tabla_a.py --resultados sendoa/resultados_oficiales --salida sendoa/tablas --solo-oficiales
boe: 3 filas -> /home/g6/reto_1_2-2627-Rag-Benchmark/sendoa/tablas/tabla_a_boe.md/.csv
debian: 3 filas -> /home/g6/reto_1_2-2627-Rag-Benchmark/sendoa/tablas/tabla_a_debian.md/.csv
 
# Tabla A - Escenario BOE

Generado: 2026-11-22T21:28:07
Filas: 3

| Modelo | Estado | Parametros | Dimension | Recall@1 | Recall@3 | Recall@5 | MRR | Indexado (s) | Indice (MB) | VRAM (GB) | Fichero origen |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Qwen/Qwen3-Embedding-0.6B | OFICIAL | 595776512 | 1024 | 0.417 | 0.667 | 1.000 | 0.833 | 3.38 | 1.05 | 2.442 | sendoa/resultados_oficiales/Qwen_Qwen3-Embedding-0.6B__boe__20261122-212645.json |
| sendoa/modelos/embeddinggemma-300m | OFICIAL | 307581696 | 768 | 0.417 | 0.917 | 0.917 | 0.847 | 3.82 | 0.79 | 2.489 | sendoa/resultados_oficiales/sendoa_modelos_embeddinggemma-300m_

## 3. Fase 3: LLMs
Se pasa el embedding ganador de cada escenario. Mismo prompt (`comun/prompt.py`), top-k 5, temperatura 0, 300 tokens de salida, 1 calentamiento + (1 descartada + 3) repeticiones en 5 preguntas.

In [13]:
LLMS = ['Qwen/Qwen3-1.7B','HuggingFaceTB/SmolLM3-3B','Qwen/Qwen3-8B','mistralai/Ministral-3-8B-Instruct-2512']
GAN = json.load(open(RAIZ/'nahia'/'ganadores_embedding.json'))
CFG = {'Qwen/Qwen3-Embedding-0.6B':'../sendoa/config_qwen3.json','sendoa/modelos/embeddinggemma-300m':'../sendoa/config_gemma.json','BAAI/bge-m3':'../sendoa/config_bge_m3.json'}
for llm in LLMS:
    for esc in ('boe','debian'):
        emb = GAN[esc]; emb_arg = ('../'+emb) if emb.startswith('sendoa/') else emb
        sh(f'{PY} benchmark_llm.py --modelo {llm} --escenario {esc} --embedding {emb_arg} --config-embedding {CFG[emb]} --turno-exclusivo', cwd=RAIZ/'nahia')

$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python benchmark_llm.py --modelo Qwen/Qwen3-1.7B --escenario boe --embedding Qwen/Qwen3-Embedding-0.6B --config-embedding ../sendoa/config_qwen3.json --turno-exclusivo
q_boe_01 rep0: 1.537 s, 48.9 tok/s, prompt 1181 tok
q_boe_01 rep1: 1.51 s, 49.8 tok/s, prompt 1181 tok
q_boe_01 rep2: 1.476 s, 51.0 tok/s, prompt 1181 tok
q_boe_01 rep3: 1.47 s, 51.2 tok/s, prompt 1181 tok
q_boe_02 rep0: 1.174 s, 51.2 tok/s, prompt 1890 tok
q_boe_02 rep1: 1.174 s, 51.2 tok/s, prompt 1890 tok
q_boe_02 rep2: 1.175 s, 51.1 tok/s, prompt 1890 tok
q_boe_02 rep3: 1.177 s, 51.1 tok/s, prompt 1890 tok
q_boe_03 rep1: 1.379 s, 50.9 tok/s, prompt 2230 tok
q_boe_04 rep1: 1.196 s, 50.4 tok/s, prompt 2263 tok
q_boe_05 rep1: 0.263 s, 56.4 tok/s, prompt 2194 tok
q_boe_06 rep1: 1.211 s, 51.2 tok/s, prompt 1806 tok
q_boe_07 rep1: 1.107 s, 50.9 tok/s, prompt 2289 tok
q_boe_08 rep1: 1.094 s, 50.9 tok/s, prompt 1709 tok
q_boe_09 rep0: 1.691 s, 50.9 tok/s, prompt 1806 tok
q_boe

## 4. Puntuación a mano (humana, cruzada)
Se generan las vistas de revisión; las notas las pone una persona en cada `nahia/resultados/*_puntuacion.csv` (criterio en `nahia/PROPUESTAS_para_el_grupo.md`). Cruzada: Julen→Mikel, Mikel→Sendoa, Sendoa→Nahia, Nahia→Julen.

In [14]:
import glob
for f in sorted(glob.glob(str(RAIZ/'nahia'/'resultados'/'*.jsonl'))):
    if 'prueba' not in f: sh(f'{PY} nahia/ayuda_puntuar.py {f}')

$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python nahia/ayuda_puntuar.py /home/g6/reto_1_2-2627-Rag-Benchmark/nahia/resultados/HuggingFaceTB_SmolLM3-3B__boe__20261122-134601.jsonl
Creado /home/g6/reto_1_2-2627-Rag-Benchmark/nahia/revision/HuggingFaceTB_SmolLM3-3B__boe__20261122-134601_revision.md
 
$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python nahia/ayuda_puntuar.py /home/g6/reto_1_2-2627-Rag-Benchmark/nahia/resultados/HuggingFaceTB_SmolLM3-3B__boe__20261122-155026.jsonl
Creado /home/g6/reto_1_2-2627-Rag-Benchmark/nahia/revision/HuggingFaceTB_SmolLM3-3B__boe__20261122-155026_revision.md
 
$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python nahia/ayuda_puntuar.py /home/g6/reto_1_2-2627-Rag-Benchmark/nahia/resultados/HuggingFaceTB_SmolLM3-3B__boe__20261122-204129.jsonl
Creado /home/g6/reto_1_2-2627-Rag-Benchmark/nahia/revision/HuggingFaceTB_SmolLM3-3B__boe__20261122-204129_revision.md
 
$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python nahia/ayuda_puntuar.py /ho

## 4b. Reutilizar las notas validadas
Con temperatura 0 las respuestas repetidas son idénticas. Este paso copia las notas ya validadas a las ejecuciones nuevas **solo si el texto de la respuesta coincide letra por letra**; si no, la fila queda vacía y hay que puntuarla a mano.

**Validación a ciegas (obligatoria si se usa un LLM como ayudante):** una persona puntúa a mano `nahia/validacion_10/validacion_ciega.csv` (10 respuestas sin notas previas) y se ejecuta `nahia/comparar_validacion.py` para anotar cuántas coinciden.

In [15]:
sh(f'{PY} nahia/copiar_puntuacion.py')

$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python nahia/copiar_puntuacion.py
HuggingFaceTB/SmolLM3-3B boe: 15/15 notas copiadas de HuggingFaceTB_SmolLM3-3B__boe__20261122-
HuggingFaceTB/SmolLM3-3B debian: 15/15 notas copiadas de HuggingFaceTB_SmolLM3-3B__debian__202611
Qwen/Qwen3-1.7B boe: 15/15 notas copiadas de Qwen_Qwen3-1.7B__boe__20261122-210639_me
Qwen/Qwen3-1.7B debian: 15/15 notas copiadas de Qwen_Qwen3-1.7B__debian__20261122-210814
Qwen/Qwen3-8B boe: 15/15 notas copiadas de Qwen_Qwen3-8B__boe__20261122-204343_meta
Qwen/Qwen3-8B debian: 15/15 notas copiadas de Qwen_Qwen3-8B__debian__20261122-203013_m
mistralai/Ministral-3-8B-Instruct-2512 boe: 15/15 notas copiadas de mistralai_Ministral-3-8B-Instruct-2512__
mistralai/Ministral-3-8B-Instruct-2512 debian: 15/15 notas copiadas de mistralai_Ministral-3-8B-Instruct-2512__
 


## 5. Tablas B y C y decisión (Fase 4)
Solo funciona cuando las hojas están puntuadas. `--vram-max 8` es el plan B.

In [16]:
for f in sorted(glob.glob(str(RAIZ/'nahia'/'resultados'/'*_meta.json'))):
    try: sh(f'{PY} nahia/tabla_bc.py {f}')
    except Exception as e: print('sin puntuar o error:', f)
sh(f'{PY} nahia/decision_fase4.py')
sh(f'{PY} nahia/decision_fase4.py --vram-max 8')

$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python nahia/tabla_bc.py /home/g6/reto_1_2-2627-Rag-Benchmark/nahia/resultados/HuggingFaceTB_SmolLM3-3B__boe__20261122-134601_meta.json
# HuggingFaceTB/SmolLM3-3B · boe · 2026-11-22T13:48:08

## Tabla B (calidad)
| Modelo | Tamaño | Cuantización | Corrección media (0–2) | Respuestas fieles (%) | Abstención correcta (de 3) | Falsas abstenciones | Formato e idioma (%) |
|---|---|---|---|---|---|---|---|
| HuggingFaceTB/SmolLM3-3B | 3.08 B | bf16 | 1.4 | 60 | 2 | 0 (de 12) | 73 |

## Tabla C (coste)
| Modelo | Stack | TTFT (s) | Tokens/s | Latencia total (s) | VRAM pico (GB) |
|---|---|---|---|---|---|
| HuggingFaceTB/SmolLM3-3B | transformers 5.23.0 | 0.17 | 52.6 | 3.3 | 6.8 |

## Trazabilidad
- Respuestas y tiempos brutos: `HuggingFaceTB_SmolLM3-3B__boe__20261122-134601.jsonl`
- Puntuación a mano (15 respuestas, puntuado por: Nahia (validación manual; pre-puntuación asistida por LLM)): `HuggingFaceTB_SmolLM3-3B__boe__20261122-134601_puntu

## 6. Tablas consolidadas por escenario y copia a comun/resultados_finales

In [17]:
sh(f'{PY} nahia/consolidar.py')

$ /home/g6/reto_1_2-2627-Rag-Benchmark/.venv/bin/python nahia/consolidar.py
OK: 8 ejecuciones finales
 
